# Inference Bioinspired3D - Finetuned Model for Blender Code Generation

Rachel K. Luu, 2026

# Load Model

In [ ]:
from scripts.inference.config import Bio3DConfig, RAGConfig
from scripts.inference.llm_bio3d import Bio3D
from scripts.inference import utils

rag_cfg = RAGConfig(
    enabled=True, # toggle here
    jsonl_paths=["../data/rag/biodataset_RAG.jsonl"],
    top_k=2,
    embed_model_name="BAAI/bge-small-en-v1.5",
)

bio3d = Bio3D(
    llm_cfg=Bio3DConfig(
        device="cuda:0",
        base_model="meta-llama/Llama-3.2-3B-Instruct",
        lora_adapter="rachelkluu/bioinspired3D",
        torch_dtype="float16",
    ),
    rag_cfg=rag_cfg,
).load()


# Inference

In [ ]:
#bio3d.set_rag(False) # use to toggle RAG on and off

raw = bio3d.generate_code(
    "Write Blender Python code to make a tubular material.",
    mode="direct",
)

resp = utils.extract_assistant_text(raw)
print(resp)



# Inference with Blender Validation

In [ ]:
import os
from datetime import datetime

from scripts.inference.utils import extract_assistant_text, extract_blender_code, clean_blender_code
from scripts.inference.blender_exec import BlenderValidator, BlenderExecConfig

# Config
WSL_RENDER_BASE = "/home/rachel/bio3d_renders" # PATH TO WHERE YOU WANT THE OUTPUT

blender = BlenderValidator(BlenderExecConfig(
    blender_path="/mnt/c/Users/Rachel/Downloads/blender-4.2.1-windows-x64/blender-4.2.1-windows-x64/blender.exe",
    win_render_base=r"C:\Users\Rachel\bio3d_tmp\renders",
    wsl_render_base=WSL_RENDER_BASE,
))

def run_bio3d_once(
    bio3d,
    prompt: str,
    *,
    mode: str = "direct",                # "direct" or "design"
    rag: bool | None = None,             # None = keep current, True/False = toggle
    label: str = "infer",
    render_subdir_wsl: str | None = None,
    max_new_tokens: int = 2048,
    temperature: float = 0.1,
    top_p: float = 0.9,
    do_sample: bool = True,
):
    if rag is not None:
        bio3d.set_rag(rag)

    # 1) Generate
    raw = bio3d.generate_code(
        prompt,
        mode=mode,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
    )

    # 2) Keep only assistant text (nice for printing/logging)
    assistant_only = extract_assistant_text(raw)

    # 3) Extract + clean Blender code
    code = clean_blender_code(extract_blender_code(assistant_only))

    # 4) Run Blender validation + render
    if render_subdir_wsl is None:
        render_subdir_wsl = os.path.join(WSL_RENDER_BASE, datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))

    result = blender.run(code, label=label, render_subdir_wsl=render_subdir_wsl)

    return {
        "assistant_text": assistant_only,
        "blender_code": code,
        "blender_result": result,
        "render_subdir_wsl": render_subdir_wsl,
    }


In [ ]:
out = run_bio3d_once(
    bio3d,
    "Write Blender Python code to make a tubular material.",
    mode="direct",
    rag=False,
    label="tubular",
)

print(out["assistant_text"])
print("\n---\nSTATUS:", out["blender_result"]["status"])
print("RENDER:", out["blender_result"]["render_path"])
print("ERROR:", out["blender_result"]["error_snippet"])
